In [1]:
!pip install feedparser pandas
!pip install pyvi

In [2]:
import feedparser
import pandas as pd
from bs4 import BeautifulSoup
from google.cloud import storage
from google.cloud import bigquery
from datetime import datetime
import time
import io
import json
import yaml # THÊM THƯ VIỆN ĐỌC YAML
from pyvi import ViTokenizer

def job_incremental_crawler():
    BUCKET_NAME = "phanthanhsum"
    JSON_CHINH_PATH = "data_lakehouse/news_chinh.json"
    STOPWORD_PATH = "stop_words_Vietnamese.txt"
    YAML_CONFIG_PATH = "sources.yaml" # Đường dẫn file cấu hình trên GCS

    TABLE_ID = "project-ceb4f683-ad1a-44e3-8d8.bigdata_project.vnexpress_world_news"

    storage_client = storage.Client()
    bucket = storage_client.bucket(BUCKET_NAME)
    blob_chinh = bucket.blob(JSON_CHINH_PATH)

    # ------------------------------------------------------------------------
    # BƯỚC 0.1: Tải và phân tích file cấu hình YAML từ GCS
    # ------------------------------------------------------------------------
    blob_yaml = bucket.blob(YAML_CONFIG_PATH)
    if not blob_yaml.exists():
        print(f"[-] Lỗi chí mạng: Không tìm thấy file cấu hình {YAML_CONFIG_PATH} trên GCS. Dừng job.")
        return

    yaml_content = blob_yaml.download_as_text(encoding='utf-8')
    # PyYAML sẽ chuyển file YAML thành một Dictionary trong Python
    RSS_SOURCES = yaml.safe_load(yaml_content)
    print(f"[*] Đã tải cấu hình thành công. Hệ thống có {len(RSS_SOURCES)} chủ đề phân tích.")

    # ------------------------------------------------------------------------
    # BƯỚC 1: Tải danh sách Stopwords
    # ------------------------------------------------------------------------
    blob_stopword = bucket.blob(STOPWORD_PATH)
    stopwords = set()
    if blob_stopword.exists():
        sw_content = blob_stopword.download_as_text(encoding='utf-8')
        stopwords = set([line.strip().lower() for line in sw_content.splitlines() if line.strip()])
        print(f"[*] Đã tải thành công {len(stopwords)} từ khóa stopword.")
    else:
        print("[-] Cảnh báo: Không tìm thấy file stopword trên GCS.")

    # ------------------------------------------------------------------------
    # BƯỚC 2: Đọc Link cũ (JSON) để chống trùng
    # ------------------------------------------------------------------------
    existing_links = set()
    if blob_chinh.exists():
        print("[*] Đã tìm thấy file news_chinh.json. Đang đối chiếu dữ liệu...")
        content = blob_chinh.download_as_text()
        df_old = pd.read_json(io.StringIO(content), lines=True)
        if "link" in df_old.columns:
            existing_links = set(df_old["link"].tolist())
    else:
        print("[*] LẦN CHẠY ĐẦU TIÊN: File chính chưa tồn tại. Hệ thống sẽ cào toàn bộ.")

    # ------------------------------------------------------------------------
    # BƯỚC 3: Cào RSS Đa chuyên mục & Đa nguồn link (Xử lý mảng YAML)
    # ------------------------------------------------------------------------
    new_clean_items = []

    # Vòng lặp 1: Duyệt qua từng Chủ đề (Ví dụ: Kinh doanh, Thế giới...)
    for category_name, url_list in RSS_SOURCES.items():
        print(f"\n======== [📂 Chủ đề: {category_name}] ========")

        # Kiểm tra tính hợp lệ của dữ liệu YAML (phải là một danh sách các link)
        if not isinstance(url_list, list):
            print(f"[-] Sai định dạng YAML cho mục {category_name}. Phải là danh sách dạng gạch đầu dòng.")
            continue

        # Vòng lặp 2: Duyệt qua từng đường link RSS có trong chủ đề đó
        for rss_url in url_list:
            print(f"   └─ Đang cào nguồn: {rss_url}")
            try:
                feed = feedparser.parse(rss_url)

                for entry in feed.entries:
                    if entry.link not in existing_links:
                        raw_html = entry.summary if hasattr(entry, 'summary') else ""
                        soup = BeautifulSoup(raw_html, "html.parser")
                        clean_desc = soup.get_text(strip=True)

                        if hasattr(entry, 'published_parsed'):
                            dt_obj = datetime.fromtimestamp(time.mktime(entry.published_parsed))
                            formatted_date = dt_obj.strftime("%Y-%m-%d")
                        else:
                            formatted_date = datetime.now().strftime("%Y-%m-%d")

                        text_to_tokenize = f"{entry.title} {clean_desc}"
                        tokenized_str = ViTokenizer.tokenize(text_to_tokenize)

                        words = tokenized_str.split()
                        clean_keywords = []
                        for w in words:
                            w_lower = w.lower()
                            if w_lower not in stopwords and (w.isalpha() or '_' in w):
                                clean_keywords.append(w)

                        # Đóng gói dữ liệu
                        new_clean_items.append({
                            "title": entry.title,
                            "publish_date": formatted_date,
                            "link": entry.link,
                            "description": clean_desc,
                            "category": category_name, # Gán tên chủ đề tổng quát từ file YAML
                            "clean_keywords": clean_keywords
                        })

                        # Đưa vào set chặn trùng nội bộ ngay lập tức
                        existing_links.add(entry.link)
            except Exception as e:
                print(f"   └─ [❌ LỖI] Không thể cào nguồn {rss_url}. Chi tiết: {e}")

    # ------------------------------------------------------------------------
    # BƯỚC 4: Kiểm tra và Xuất bản lên BigQuery / GCS
    # ------------------------------------------------------------------------
    if not new_clean_items:
        print("\n[-] Không có bài viết nào mới trên tất cả các nguồn RSS. Dừng tiến trình nạp.")
        return

    df_tam = pd.DataFrame(new_clean_items)
    print(f"\n[+] Tổng cộng tìm thấy {len(df_tam)} bài viết mới tinh. Đang tiến hành nạp vào BigQuery...")

    # Lưu file TẠM định dạng NDJSON
    current_time = datetime.now().strftime("%Y%m%d_%H%M")
    JSON_TAM_PATH = f"staging_zone/news_tam_{current_time}.json"
    blob_tam = bucket.blob(JSON_TAM_PATH)

    json_data = df_tam.to_json(orient="records", lines=True, force_ascii=False)
    blob_tam.upload_from_string(json_data, content_type='application/json')
    print(f"[+] Đã đẩy file tạm lên GCS: {JSON_TAM_PATH}")

    # Ra lệnh cho BigQuery nạp file JSON tạm
    bq_client = bigquery.Client()
    job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
        autodetect=True,
        write_disposition="WRITE_APPEND",
        schema_update_options=[bigquery.SchemaUpdateOption.ALLOW_FIELD_ADDITION]
    )

    gcs_uri_tam = f"gs://{BUCKET_NAME}/{JSON_TAM_PATH}"
    print(f"[*] Đang yêu cầu BigQuery APPEND dữ liệu vào bảng {TABLE_ID}...")
    load_job = bq_client.load_table_from_uri(gcs_uri_tam, TABLE_ID, job_config=job_config)
    load_job.result()
    print("[🎉] BigQuery đã nạp thành công!")

    # Gộp file tạm vào file chính (JSON) trên GCS làm Backup Warehouse
    if blob_chinh.exists():
        df_chinh_old = pd.read_json(io.StringIO(blob_chinh.download_as_text()), lines=True)
        df_chinh_new = pd.concat([df_chinh_old, df_tam], ignore_index=True)
    else:
        df_chinh_new = df_tam

    json_chinh_data = df_chinh_new.to_json(orient="records", lines=True, force_ascii=False)
    blob_chinh.upload_from_string(json_chinh_data, content_type='application/json')
    print(f"[🎉] Đã lưu cập nhật file gốc tại: {JSON_CHINH_PATH}")

if __name__ == "__main__":
    job_incremental_crawler()
    print('Đã xong')

[*] Đã tải cấu hình thành công. Hệ thống có 16 chủ đề phân tích.
[*] Đã tải thành công 75 từ khóa stopword.
[*] Đã tìm thấy file news_chinh.json. Đang đối chiếu dữ liệu...

======== [📂 Chủ đề: Thế giới] ========
   └─ Đang cào nguồn: https://vnexpress.net/rss/the-gioi.rss

======== [📂 Chủ đề: Thời sự] ========
   └─ Đang cào nguồn: https://vnexpress.net/rss/thoi-su.rss

======== [📂 Chủ đề: Kinh doanh] ========
   └─ Đang cào nguồn: https://vnexpress.net/rss/kinh-doanh.rss

======== [📂 Chủ đề: Giải trí] ========
   └─ Đang cào nguồn: https://vnexpress.net/rss/giai-tri.rss

======== [📂 Chủ đề: Thể thao] ========
   └─ Đang cào nguồn: https://vnexpress.net/rss/the-thao.rss

======== [📂 Chủ đề: Pháp luật] ========
   └─ Đang cào nguồn: https://vnexpress.net/rss/phap-luat.rss

======== [📂 Chủ đề: Giáo dục] ========
   └─ Đang cào nguồn: https://vnexpress.net/rss/giao-duc.rss

======== [📂 Chủ đề: Bất động sản] ========
   └─ Đang cào nguồn: https://vnexpress.net/rss/bat-dong-san.rss

=======

In [ ]:
-- Bước 1: Khai báo biến chọn Chuyên mục tại đây.
-- Bạn có thể đổi thành 'Kinh doanh', 'Thể thao', 'Giáo dục'... hoặc điền 'Tất cả' để lấy hết toàn bộ.
DECLARE target_category STRING DEFAULT 'Tất cả';

-- Bước 2: Chạy câu lệnh truy vấn tìm xu hướng
SELECT
  keyword AS Tu_Khoa,
  COUNT(*) AS So_Lan_Xuat_Hien
FROM
  `project-ceb4f683-ad1a-44e3-8d8.bigdata_project.vnexpress_world_news`,
  UNNEST(clean_keywords) AS keyword
WHERE
  -- Lấy các bài báo xuất bản trong vòng 2 ngày qua
  publish_date >= DATE_SUB(CURRENT_DATE('Asia/Ho_Chi_Minh'), INTERVAL 2 DAY)

  -- Mẹo logic linh hoạt:
  -- Nếu điền chuyên mục cụ thể -> vế đầu đúng. Nếu điền 'Tất cả' -> vế sau đúng (bỏ qua bộ lọc).
  AND (category = target_category OR target_category = 'Tất cả')
GROUP BY
  keyword
ORDER BY
  So_Lan_Xuat_Hien DESC
LIMIT 15;

,Tu_Khoa,So_Lan_Xuat_Hien
0,trong,111
1,ở,109
2,người,106
3,năm,66
4,Mỹ,59
5,một,57
6,về,55
7,nước,51
8,triệu,46
9,tuổi,43


**SQL tìm cụm 2 từ**

In [ ]:
# sql_engine: bigquery
# output_variable: df
# output_mode: table
# start _sql
_sql = """
-- Bước 1: Khai báo biến chọn Chuyên mục
-- Bạn có thể đổi thành 'Kinh doanh', 'Thế giới', 'Giáo dục'... hoặc điền 'Tất cả'
DECLARE target_category STRING DEFAULT 'Tất cả';

WITH Bigram_Base AS (
  SELECT
    -- Ghép từ ở vị trí hiện tại (i) với từ ở vị trí kế tiếp (i + 1)
    clean_keywords[OFFSET(i)] || ' ' || clean_keywords[OFFSET(i + 1)] AS Cum_2_Tu
  FROM
    `project-ceb4f683-ad1a-44e3-8d8.bigdata_project.vnexpress_world_news`,
    -- Rã mảng ra nhưng giữ lại số thứ tự (OFFSET) của từng từ
    UNNEST(clean_keywords) WITH OFFSET i
  WHERE
    -- Giới hạn để vòng lặp không bị tràn mảng (Out of bounds)
    i < ARRAY_LENGTH(clean_keywords) - 1

    -- Lọc dữ liệu trong 2 ngày qua
    AND publish_date >= DATE_SUB(CURRENT_DATE('Asia/Ho_Chi_Minh'), INTERVAL 2 DAY)

    -- BỘ LỌC CHUYÊN MỤC ĐỘNG: Được đặt ngay tại bước cào dữ liệu gốc
    AND (category = target_category OR target_category = 'Tất cả')
)

-- Bước 2: Bảng kết quả đếm tần suất cụm 2 từ
SELECT
  Cum_2_Tu,
  COUNT(*) AS So_Lan_Xuat_Hien
FROM
  Bigram_Base
GROUP BY
  Cum_2_Tu
ORDER BY
  So_Lan_Xuat_Hien DESC
LIMIT 15;
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
df = _bqsqlcell.run(_sql)
df

,Cum_2_Tu,So_Lan_Xuat_Hien
0,TP HCM,26
1,triệu đồng,20
2,tỷ đồng,18
3,Tổng_Bí_thư Chủ_tịch,12
4,Chủ_tịch nước,12
5,triệu USD,11
6,người dân,10
7,thi lớp,10
8,độ C,9
9,châu Á,9


**SQL tìm cụm 3 từ**

In [ ]:
# sql_engine: bigquery
# output_variable: df
# output_mode: table
# start _sql
_sql = """
-- Bước 1: Khai báo biến chọn Chuyên mục
-- Bạn có thể đổi thành 'Kinh doanh', 'Thế giới', 'Giáo dục'... hoặc điền 'Tất cả'
DECLARE target_category STRING DEFAULT 'Tất cả';

WITH Trigram_Base AS (
  SELECT
    -- Ghép 3 từ liên tiếp lại với nhau
    clean_keywords[OFFSET(i)] || ' ' || clean_keywords[OFFSET(i + 1)] || ' ' || clean_keywords[OFFSET(i + 2)] AS Cum_3_Tu
  FROM
    `project-ceb4f683-ad1a-44e3-8d8.bigdata_project.vnexpress_world_news`,
    UNNEST(clean_keywords) WITH OFFSET i
  WHERE
    -- Giới hạn để vòng lặp không bị tràn mảng (Out of bounds)
    i < ARRAY_LENGTH(clean_keywords) - 2

    -- Lọc dữ liệu trong 2 ngày qua
    AND publish_date >= DATE_SUB(CURRENT_DATE('Asia/Ho_Chi_Minh'), INTERVAL 2 DAY)

    -- BỘ LỌC CHUYÊN MỤC ĐỘNG
    AND (category = target_category OR target_category = 'Tất cả')
)

-- Bước 2: Bảng kết quả đếm tần suất cụm 3 từ
SELECT
  Cum_3_Tu,
  COUNT(*) AS So_Lan_Xuat_Hien
FROM
  Trigram_Base
GROUP BY
  Cum_3_Tu
ORDER BY
  So_Lan_Xuat_Hien DESC
LIMIT 15;
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
df = _bqsqlcell.run(_sql)
df

,Cum_3_Tu,So_Lan_Xuat_Hien
0,Tổng_Bí_thư Chủ_tịch nước,12
1,Chủ_tịch nước Tô_Lâm,9
2,ở vòng hai,4
3,Toán chuyên thi,4
4,cánh vũ_trang Hamas,4
5,nhà ở xã_hội,4
6,chuyên thi lớp,4
7,nước Tô_Lâm phu_nhân,4
8,trong mùa hè,4
9,Tổng_thống Trump tuyên_bố,3


**TF-IDF**

In [ ]:
# sql_engine: bigquery
# output_variable: df
# output_mode: table
# start _sql
_sql = """
DECLARE target_category STRING DEFAULT 'Tất cả';

WITH Word_Counts AS (
  SELECT link, keyword, COUNT(*) AS tf_count
  FROM `project-ceb4f683-ad1a-44e3-8d8.bigdata_project.vnexpress_world_news`,
  UNNEST(clean_keywords) AS keyword
  WHERE publish_date >= DATE_SUB(CURRENT_DATE('Asia/Ho_Chi_Minh'), INTERVAL 2 DAY)
  AND (category = target_category OR target_category = 'Tất cả')
  GROUP BY link, keyword
),
Total_Docs AS (
  SELECT COUNT(DISTINCT link) AS total_docs FROM Word_Counts
),
Doc_Frequency AS (
  SELECT keyword, COUNT(DISTINCT link) AS doc_count FROM Word_Counts GROUP BY keyword
)

SELECT
  w.keyword AS Tu_Khoa,
  SUM(w.tf_count) AS Tong_So_Lan_Xuat_Hien,
  -- 1. CÔNG THỨC TOÁN HỌC CHUẨN: Dùng Log10 để hãm sức mạnh của TF lại
  ROUND(SUM( (1 + LOG10(w.tf_count)) * LOG10(t.total_docs / d.doc_count) ), 2) AS Diem_Trend_TF_IDF
FROM
  Word_Counts w
JOIN Doc_Frequency d ON w.keyword = d.keyword
CROSS JOIN Total_Docs t
WHERE
  -- 2. NGƯỠNG LOẠI TRỪ (Max DF Threshold):
  -- Nếu từ khóa xuất hiện ở NHIỀU HƠN 25% tổng số bài báo -> Đích thị là từ rác (như "trong", "ở") -> XÓA!
  (d.doc_count / t.total_docs) <= 0.25

  -- 3. BỘ LỌC CHỮA CHÁY (Loại bỏ các từ cơ bản bị lọt lưới)
  AND LENGTH(w.keyword) >= 3 -- Bỏ các từ có 1-2 chữ cái (ở, là, có, vì, và...)
  AND LOWER(w.keyword) NOT IN ('người', 'năm', 'nước', 'nhà', 'một', 'việt_nam', 'ngày', 'của', 'được', 'với', 'những', 'các', 'cho')
GROUP BY
  w.keyword
ORDER BY
  Diem_Trend_TF_IDF DESC
LIMIT 15;
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
df = _bqsqlcell.run(_sql)
df

,Tu_Khoa,Tong_So_Lan_Xuat_Hien,Diem_Trend_TF_IDF
0,trong,111,63.06
1,tuổi,43,40.77
2,triệu,46,39.08
3,hơn,41,38.73
4,nhất,39,38.05
5,khiến,38,38.03
6,đồng,40,37.45
7,hai,36,37.06
8,tháng,34,33.63
9,đến,29,32.58


**TF-IDF cho 2 từ**

In [ ]:
# sql_engine: bigquery
# output_variable: df
# output_mode: table
# start _sql
_sql = """
DECLARE target_category STRING DEFAULT 'Tất cả';

-- 1. Tạo bảng Cụm 2 từ trước
WITH Bigram_Base AS (
  SELECT
    link,
    -- Ghép 2 từ liên tiếp để tạo cụm (Bigram)
    clean_keywords[OFFSET(i)] || ' ' || clean_keywords[OFFSET(i + 1)] AS bigram
  FROM
    `project-ceb4f683-ad1a-44e3-8d8.bigdata_project.vnexpress_world_news`,
    UNNEST(clean_keywords) WITH OFFSET i
  WHERE
    i < ARRAY_LENGTH(clean_keywords) - 1
    AND publish_date >= DATE_SUB(CURRENT_DATE('Asia/Ho_Chi_Minh'), INTERVAL 2 DAY)
    AND (category = target_category OR target_category = 'Tất cả')
),
-- 2. Đưa cụm 2 từ vào tính TF
Word_Counts AS (
  SELECT link, bigram AS keyword, COUNT(*) AS tf_count
  FROM Bigram_Base
  GROUP BY link, bigram
),
Total_Docs AS (
  SELECT COUNT(DISTINCT link) AS total_docs FROM Word_Counts
),
Doc_Frequency AS (
  SELECT keyword, COUNT(DISTINCT link) AS doc_count FROM Word_Counts GROUP BY keyword
)

-- 3. Tính điểm TF-IDF cho Cụm 2 từ
SELECT
  w.keyword AS Cum_2_Tu_Hot,
  SUM(w.tf_count) AS Tong_So_Lan_Xuat_Hien,
  ROUND(SUM( (1 + LOG10(w.tf_count)) * LOG10(t.total_docs / d.doc_count) ), 2) AS Diem_Trend_TF_IDF
FROM
  Word_Counts w
JOIN Doc_Frequency d ON w.keyword = d.keyword
CROSS JOIN Total_Docs t
WHERE
  -- Lọc bớt các cụm rác ghép ngẫu nhiên (VD: "trong một", "đến nay") xuất hiện > 15% số bài
  (d.doc_count / t.total_docs) <= 0.15
GROUP BY
  w.keyword
ORDER BY
  Diem_Trend_TF_IDF DESC
LIMIT 15;
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
df = _bqsqlcell.run(_sql)
df

,Cum_2_Tu_Hot,Tong_So_Lan_Xuat_Hien,Diem_Trend_TF_IDF
0,TP HCM,26,28.58
1,triệu đồng,20,23.17
2,tỷ đồng,18,22.28
3,người dân,10,15.42
4,triệu USD,11,15.21
5,Tổng_Bí_thư Chủ_tịch,12,15.03
6,Chủ_tịch nước,12,15.03
7,châu Âu,9,13.43
8,thi lớp,10,13.21
9,châu Á,9,12.66


**TF-IDF cho 3 từ**

In [ ]:
# sql_engine: bigquery
# output_variable: df
# output_mode: table
# start _sql
_sql = """
DECLARE target_category STRING DEFAULT 'Tất cả';

-- 1. Tạo bảng Cụm 3 từ (Trigram_Base)
WITH Trigram_Base AS (
  SELECT
    link,
    -- NÂNG CẤP: Ghép 3 từ liên tiếp (i, i+1, i+2)
    clean_keywords[OFFSET(i)] || ' ' || clean_keywords[OFFSET(i + 1)] || ' ' || clean_keywords[OFFSET(i + 2)] AS trigram
  FROM
    `project-ceb4f683-ad1a-44e3-8d8.bigdata_project.vnexpress_world_news`,
    UNNEST(clean_keywords) WITH OFFSET i
  WHERE
    -- NÂNG CẤP: Phải trừ đi 2 để vòng lặp không báo lỗi Out of bounds khi lấy (i+2)
    i < ARRAY_LENGTH(clean_keywords) - 2
    AND publish_date >= DATE_SUB(CURRENT_DATE('Asia/Ho_Chi_Minh'), INTERVAL 2 DAY)
    AND (category = target_category OR target_category = 'Tất cả')
),
-- 2. Đưa cụm 3 từ vào tính TF
Word_Counts AS (
  SELECT link, trigram AS keyword, COUNT(*) AS tf_count
  FROM Trigram_Base
  GROUP BY link, trigram
),
Total_Docs AS (
  SELECT COUNT(DISTINCT link) AS total_docs FROM Word_Counts
),
Doc_Frequency AS (
  SELECT keyword, COUNT(DISTINCT link) AS doc_count FROM Word_Counts GROUP BY keyword
)

-- 3. Tính điểm TF-IDF cho Cụm 3 từ
SELECT
  w.keyword AS Cum_3_Tu_Hot,
  SUM(w.tf_count) AS Tong_So_Lan_Xuat_Hien,
  ROUND(SUM( (1 + LOG10(w.tf_count)) * LOG10(t.total_docs / d.doc_count) ), 2) AS Diem_Trend_TF_IDF
FROM
  Word_Counts w
JOIN Doc_Frequency d ON w.keyword = d.keyword
CROSS JOIN Total_Docs t
WHERE
  -- ĐIỀU CHỈNH NGƯỠNG LỌC: Cụm 3 từ rất dài nên hiếm khi trùng lặp nhiều.
  -- Thậm chí bạn có thể bỏ luôn cái WHERE d.doc_count... này đi,
  -- nhưng nếu giữ thì hãy nâng lên cao để an toàn (VD: <= 0.5)
  (d.doc_count / t.total_docs) <= 0.5
GROUP BY
  w.keyword
ORDER BY
  Diem_Trend_TF_IDF DESC
LIMIT 15;
""" # end _sql
from google.colab.sql import bigquery as _bqsqlcell
df = _bqsqlcell.run(_sql)
df

,Cum_3_Tu_Hot,Tong_So_Lan_Xuat_Hien,Diem_Trend_TF_IDF
0,Tổng_Bí_thư Chủ_tịch nước,12,15.03
1,Chủ_tịch nước Tô_Lâm,9,12.66
2,nước Tô_Lâm phu_nhân,4,8.04
3,ở vòng hai,4,8.04
4,trong mùa hè,4,7.05
5,nhà ở xã_hội,4,7.05
6,Toán chuyên thi,4,7.05
7,chuyên thi lớp,4,7.05
8,Tổng_thống Trump tuyên_bố,3,6.4
9,thi lớp tỉnh,3,6.4
